In [27]:
from pathlib import Path
import numpy as np
import scipy.signal
import wfdb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import resample
from functools import lru_cache
from sklearn.metrics import roc_auc_score, average_precision_score
import copy, math

FS             = 250
WINDOW_SEC     = 60
WINDOW_SAMPLES = FS * WINDOW_SEC      # 15000
PURITY_THRESH  = 0.90
BEAT_LENGTH    = 160                  # matches patch_len
MAX_BEATS      = 93                   # matches PE size
D_MODEL        = 256
N_LAYERS       = 6
N_HEADS        = 8
DROPOUT        = 0.1
P_AF_TRAIN     = 0.5                  # balanced training

ROOT           = Path("/data/ahmed/icentia11k")
CKPT_PATH      = Path("CNN_tokenizer_1min/checkpoint_50000.pth")
VALID_BEAT_SYMBOLS = {"N", "S", "V"}

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [28]:
def list_patients(root: Path):
    patients = []
    for sub in sorted(root.glob("p*/p*")):
        if sub.is_dir() and sub.name.startswith("p"):
            patients.append(sub)
    return patients


def list_record_bases(patient_dir: Path):
    recs = []
    for h in sorted(patient_dir.glob("*.hea")):
        base = h.with_suffix("")
        if base.with_suffix(".dat").exists() and base.with_suffix(".atr").exists():
            recs.append(base)
    return recs


RHY_MAP = {
    "N": "N", "AFIB": "AFIB",
    "AFL": "AFL", "AFLUT": "AFL", "AFLUTTER": "AFL",
}

def normalize_rhythm_token(tok: str):
    tok = tok.strip().upper().replace("(", "").replace(")", "").strip()
    return RHY_MAP.get(tok, None)


def build_rhythm_intervals_from_ann(ann, sig_len: int):
    intervals = []
    cur_label = None
    cur_start = None

    def close_at(end_s):
        nonlocal cur_label, cur_start, intervals
        if cur_start is not None and cur_label is not None and end_s > cur_start:
            intervals.append((int(cur_start), int(end_s), cur_label))
        cur_label = None
        cur_start = None

    for s, note in zip(ann.sample, ann.aux_note):
        if note is None:
            continue
        note = str(note).strip()
        if note == "" or note.upper() == "NONE":
            continue
        s = int(s)
        if note.startswith("("):
            if cur_start is not None:
                close_at(s)
            lab = normalize_rhythm_token(note)
            if lab is not None:
                cur_label = lab
                cur_start = s
            else:
                cur_label = None
                cur_start = None
        elif note.startswith(")"):
            close_at(s)

    if cur_start is not None and cur_label is not None:
        close_at(sig_len)

    return intervals


def window_purity_label(intervals, w_start, w_end, purity_thresh=0.90):
    dur = w_end - w_start
    occ = {"AFIB": 0, "AFL": 0, "N": 0}
    for a, b, lab in intervals:
        if b <= w_start or a >= w_end:
            continue
        overlap = max(0, min(b, w_end) - max(a, w_start))
        if lab in occ:
            occ[lab] += overlap
    covered  = sum(occ.values())
    if covered / dur < purity_thresh:
        return None
    dominant = max(occ, key=occ.get)
    if occ[dominant] / dur < purity_thresh:
        return None
    return dominant   # "N", "AFIB", or "AFL"


def split_patients(patient_dirs, ssl_frac=0.8, train_frac=0.1, val_frac=0.05, seed=42):
    rng = np.random.default_rng(seed)
    patient_dirs = list(patient_dirs)
    rng.shuffle(patient_dirs)
    n       = len(patient_dirs)
    n_ssl   = int(n * ssl_frac)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)
    ssl     = patient_dirs[:n_ssl]
    train   = patient_dirs[n_ssl : n_ssl + n_train]
    val     = patient_dirs[n_ssl + n_train : n_ssl + n_train + n_val]
    test    = patient_dirs[n_ssl + n_train + n_val:]
    return ssl, train, val, test


patients = list_patients(ROOT)
ssl_patients, train_patients, val_patients, test_patients = split_patients(patients)
print(f"SSL: {len(ssl_patients)} | Train: {len(train_patients)} | Val: {len(val_patients)} | Test: {len(test_patients)}")

SSL: 8800 | Train: 1100 | Val: 550 | Test: 550


In [29]:
# ---- Fixed patch tokenizer (same as pretrained) ----
class ConvPatchTokenizer(nn.Module):
    def __init__(self, d_model=256, patch_len=160):
        super().__init__()
        self.patch_len = patch_len
        self.conv = nn.Conv1d(1, d_model, kernel_size=patch_len, stride=patch_len)

    def forward(self, x):   # x: [B, T]
        return self.conv(x.unsqueeze(1)).transpose(1, 2)   # [B, L, d_model]


class ECGEncoder(nn.Module):
    """Fixed patch tokenizer encoder — identical to pretraining."""
    def __init__(self, d_model=256, n_layers=6, n_heads=8,
                 dropout=0.1, max_len=95, patch_len=160):
        super().__init__()
        self.tokenizer = ConvPatchTokenizer(d_model, patch_len)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=4*d_model, dropout=dropout,
            batch_first=True, activation="gelu", norm_first=True,
        )
        self.tr   = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):   # x: [B, T]
        tok = self.tokenizer(x)                          # [B, L, D]
        tok = tok + self.pos_embed[:, :tok.size(1), :]
        h   = self.norm(self.tr(tok)).mean(dim=1)        # mean pool
        return h


# ---- Beat-sync tokenizer encoder ----
class BeatSyncECGEncoder(nn.Module):
    """
    Same Conv1d kernel as ConvPatchTokenizer but applied per-beat.
    Weights are loaded from the pretrained fixed-patch checkpoint.
    """
    def __init__(self, d_model=256, n_layers=6, n_heads=8,
                 dropout=0.1, beat_length=160, max_beats=93):
        super().__init__()
        self.conv      = nn.Conv1d(1, d_model, kernel_size=beat_length, stride=beat_length)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_beats, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=4*d_model, dropout=dropout,
            batch_first=True, activation="gelu", norm_first=True,
        )
        self.tr   = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, beats, padding_mask=None):
        # beats: [B, N, 1, BEAT_LENGTH]
        B, N, C, T = beats.shape
        tok = self.conv(beats.view(B*N, C, T))           # [B*N, D, 1]
        tok = tok.squeeze(-1).view(B, N, -1)             # [B, N, D]
        tok = tok + self.pos_embed[:, :N, :]
        h   = self.norm(self.tr(tok, src_key_padding_mask=padding_mask))

        if padding_mask is not None:
            real   = (~padding_mask).unsqueeze(-1).float()
            pooled = (h * real).sum(dim=1) / real.sum(dim=1).clamp(min=1)
        else:
            pooled = h.mean(dim=1)
        return pooled

class HREncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )

    def forward(self, rr):          # rr: [B, N] in seconds
        return self.mlp(rr.unsqueeze(-1))   # [B, N, d_model]


class BeatSyncHRECGEncoder(nn.Module):
    """Beat-sync + explicit R-R interval encoding (Tok3)."""
    def __init__(self, d_model=256, n_layers=6, n_heads=8,
                 dropout=0.1, beat_length=160, max_beats=93):
        super().__init__()
        self.conv       = nn.Conv1d(1, d_model, kernel_size=beat_length, stride=beat_length)
        self.hr_encoder = HREncoder(d_model)
        self.pos_embed  = nn.Parameter(torch.zeros(1, max_beats, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=4*d_model, dropout=dropout,
            batch_first=True, activation="gelu", norm_first=True,
        )
        self.tr   = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, beats, rr_intervals, padding_mask=None):
        # beats: [B, N, 1, BEAT_LENGTH]
        # rr_intervals: [B, N] in seconds
        B, N, C, T = beats.shape
        tok = self.conv(beats.view(B*N, C, T))     # [B*N, D, 1]
        tok = tok.squeeze(-1).view(B, N, -1)       # [B, N, D]
        tok = tok + self.hr_encoder(rr_intervals)  # add R-R embedding
        tok = tok + self.pos_embed[:, :N, :]
        h   = self.norm(self.tr(tok, src_key_padding_mask=padding_mask))

        if padding_mask is not None:
            real   = (~padding_mask).unsqueeze(-1).float()
            pooled = (h * real).sum(dim=1) / real.sum(dim=1).clamp(min=1)
        else:
            pooled = h.mean(dim=1)
        return pooled


class BeatSyncHRClassifier(nn.Module):
    def __init__(self, encoder, d_model=256, num_classes=2):
        super().__init__()
        self.encoder    = encoder
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, beats, rr_intervals, padding_mask):
        return self.classifier(self.encoder(beats, rr_intervals, padding_mask))

# ---- Classifier wrappers ----
class FixedClassifier(nn.Module):
    def __init__(self, encoder, d_model=256, num_classes=2):
        super().__init__()
        self.encoder    = encoder
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        return self.classifier(self.encoder(x))


class BeatSyncClassifier(nn.Module):
    def __init__(self, encoder, d_model=256, num_classes=2):
        super().__init__()
        self.encoder    = encoder
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, beats, padding_mask):
        return self.classifier(self.encoder(beats, padding_mask))

In [30]:
ckpt       = torch.load(CKPT_PATH, map_location=device, weights_only=False)
state_dict = ckpt["model_state_dict"]
print(f"Loaded checkpoint at step {ckpt.get('step', 'unknown')}")

# ---- Fixed encoder ----
fixed_encoder = ECGEncoder(
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    dropout=DROPOUT, max_len=95, patch_len=BEAT_LENGTH,
).to(device)

# The ECGEncoder uses the same key names as the pretrained model
# but the pretrained model uses proj_dim/etc — filter to matching keys only
fixed_state = {k: v for k, v in state_dict.items()
               if k in fixed_encoder.state_dict()
               and v.shape == fixed_encoder.state_dict()[k].shape}
missing = fixed_encoder.load_state_dict(fixed_state, strict=False)
print(f"Fixed encoder — missing: {missing.missing_keys}")

fixed_model = FixedClassifier(fixed_encoder, D_MODEL, num_classes=2).to(device)

# ---- Beat-sync encoder ----
beat_encoder = BeatSyncECGEncoder(
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    dropout=DROPOUT, beat_length=BEAT_LENGTH, max_beats=MAX_BEATS,
).to(device)

# Remap tokenizer.conv.* → conv.*
remap = {}
for k, v in state_dict.items():
    if k.startswith("tokenizer.conv."):
        new_k = k.replace("tokenizer.conv.", "conv.")
        remap[new_k] = v
    else:
        remap[k] = v

remap["pos_embed"] = state_dict["pos_embed"][:, :MAX_BEATS, :]
beat_state = {k: v for k, v in remap.items()
              if k in beat_encoder.state_dict()
              and v.shape == beat_encoder.state_dict()[k].shape}
missing = beat_encoder.load_state_dict(beat_state, strict=False)
print(f"Beat-sync encoder — missing: {missing.missing_keys}")

# ---- Beat-sync + HR encoder ----
beat_hr_encoder = BeatSyncHRECGEncoder(
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    dropout=DROPOUT, beat_length=BEAT_LENGTH, max_beats=MAX_BEATS,
).to(device)

# same remap as Tok1 beat-sync
remap_hr = {}
for k, v in state_dict.items():
    if k.startswith("tokenizer.conv."):
        remap_hr[k.replace("tokenizer.conv.", "conv.")] = v
    else:
        remap_hr[k] = v
remap_hr["pos_embed"] = state_dict["pos_embed"][:, :MAX_BEATS, :]

beat_hr_state = {k: v for k, v in remap_hr.items()
                 if k in beat_hr_encoder.state_dict()
                 and v.shape == beat_hr_encoder.state_dict()[k].shape}
missing = beat_hr_encoder.load_state_dict(beat_hr_state, strict=False)
print(f"Beat-sync+HR encoder — missing: {missing.missing_keys}")
# expected missing: hr_encoder.mlp.* (random init, no pretraining)

beat_hr_model = BeatSyncHRClassifier(beat_hr_encoder, D_MODEL, num_classes=2).to(device)

beat_model = BeatSyncClassifier(beat_encoder, D_MODEL, num_classes=2).to(device)

Loaded checkpoint at step 50000
Fixed encoder — missing: []
Beat-sync encoder — missing: []
Beat-sync+HR encoder — missing: ['hr_encoder.mlp.0.weight', 'hr_encoder.mlp.0.bias', 'hr_encoder.mlp.2.weight', 'hr_encoder.mlp.2.bias']


/tmp/ipykernel_2230716/465641261.py:24: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr   = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
/tmp/ipykernel_2230716/465641261.py:51: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr   = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
/tmp/ipykernel_2230716/465641261.py:96: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr   = nn.TransformerEncoder(enc_layer, num_layers=n_layers)


In [31]:
_b, _a = scipy.signal.butter(
    4, [0.5 / (0.5*FS), 40.0 / (0.5*FS)], btype="bandpass"
)

def load_signal(rec_base_str: str):
    sig, _ = wfdb.rdsamp(rec_base_str)
    sig = sig[:, 0].astype(np.float32) if sig.ndim == 2 else sig.squeeze().astype(np.float32)
    return scipy.signal.filtfilt(_b, _a, sig).astype(np.float32)

def extract_beats_and_rr_from_ann(sig_window, ann, w_start, w_end,
                                   beat_length=160, min_beats=3):
    
    r_peaks = sorted([
        int(s) - w_start
        for s, sym in zip(ann.sample, ann.symbol)
        if w_start <= int(s) < w_end and sym in VALID_BEAT_SYMBOLS
    ])

    if len(r_peaks) < 2:
        return None, None

    beats       = []
    rr_intervals = []

    for i in range(len(r_peaks) - 1):
        start, end = r_peaks[i], r_peaks[i+1]
        beat = sig_window[start:end]
        if len(beat) < 10:
            continue
        beats.append(resample(beat, beat_length).astype(np.float32))
        rr_intervals.append((end - start) / FS)   # in seconds

    if len(beats) < min_beats:
        return None, None

    beat_arr = np.stack(beats, axis=0)[:, np.newaxis, :]   # [N, 1, beat_length]
    rr_arr   = np.array(rr_intervals, dtype=np.float32)    # [N]
    return beat_arr, rr_arr

LABEL2ID = {"N": 0, "AFIB": 1, "AFL": 1}    # binary: N=0, AF=1
    

def build_manifest(patient_dirs, purity_thresh=0.90, min_beats=3,
                   windows_per_patient=2, max_tries=200, seed=42):
    rng = np.random.default_rng(seed)
    manifest = []

    for i, pd in enumerate(patient_dirs):
        recs = list_record_bases(pd)
        if not recs:
            continue

        collected = []
        tries     = 0

        while len(collected) < windows_per_patient and tries < max_tries:
            tries += 1

            rec = recs[int(rng.integers(0, len(recs)))]
            try:
                sig = load_signal(str(rec))
                ann = wfdb.rdann(str(rec), extension="atr")
                ivs = build_rhythm_intervals_from_ann(ann, len(sig))
            except Exception:
                continue

            sig_len = len(sig)
            if sig_len <= WINDOW_SAMPLES:
                continue

            w_start = int(rng.integers(0, sig_len - WINDOW_SAMPLES))
            w_end   = w_start + WINDOW_SAMPLES

            lab = window_purity_label(ivs, w_start, w_end, purity_thresh)
            if lab is None:
                continue

            chunk = sig[w_start:w_end]
            if np.isnan(chunk).any() or np.std(chunk) < 1e-4:
                continue

            x = (chunk - chunk.mean()) / (chunk.std() + 1e-6)
            beats, rr = extract_beats_and_rr_from_ann(
                x, ann, w_start, w_end, BEAT_LENGTH, min_beats
            )
            if beats is None:
                continue

            collected.append({
                "rec":     str(rec),
                "w_start": int(w_start),
                "label":   int(LABEL2ID[lab]),
            })

        manifest.extend(collected)

        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(patient_dirs)} patients | {len(manifest)} windows so far")

    return manifest


print("Building val manifest...")
val_manifest  = build_manifest(val_patients,  windows_per_patient=2, seed=43)
print(f"Val manifest:  {len(val_manifest)} windows")

print("\nBuilding test manifest...")
test_manifest = build_manifest(test_patients, windows_per_patient=2, seed=44)
print(f"Test manifest: {len(test_manifest)} windows")

# label distribution
for name, mf in [("Val", val_manifest), ("Test", test_manifest)]:
    labels = np.array([e["label"] for e in mf])
    n_af  = (labels == 1).sum()
    n_nsr = (labels == 0).sum()
    print(f"{name}: N={len(labels)} | AF={n_af} ({100*n_af/len(labels):.1f}%) | NSR={n_nsr} ({100*n_nsr/len(labels):.1f}%)")

class IcentiaManifestDataset(Dataset):
    """
    Fixed deterministic dataset for val/test.
    All tokenizers index into the same manifest entries.
    mode: 'fixed' | 'beat' | 'beat_hr'
    """
    def __init__(self, manifest, mode="fixed", cache_size=128):
        self.manifest = manifest
        self.mode     = mode
        self._init_cache(cache_size)

    def __len__(self):
        return len(self.manifest)

    def _init_cache(self, cache_size):
        @lru_cache(maxsize=cache_size)
        def _load(rec_str):
            sig = load_signal(rec_str)
            ann = wfdb.rdann(rec_str, extension="atr")
            return sig, ann
        self._load = _load

    def __getitem__(self, idx):
        entry   = self.manifest[idx]
        sig, ann = self._load(entry["rec"])
        w_start  = entry["w_start"]
        w_end    = w_start + WINDOW_SAMPLES

        chunk = sig[w_start:w_end]
        x     = (chunk - chunk.mean()) / (chunk.std() + 1e-6)
        y     = torch.tensor(entry["label"], dtype=torch.long)

        if self.mode == "fixed":
            return torch.from_numpy(x.astype(np.float32)), y

        beats, rr = extract_beats_and_rr_from_ann(
            x, ann, w_start, w_end, BEAT_LENGTH
        )

        if self.mode == "beat":
            return torch.from_numpy(beats), y

        return torch.from_numpy(beats), torch.from_numpy(rr), y
    
class IcentiaTrainDataset(Dataset):
    """
    Training dataset with per-epoch deterministic sampling.
    Each patient gets one window per epoch, determined by:
        rng = seed + patient_idx * 1000003 + epoch * 999983
    Different epochs → different windows. Same epoch → same windows.
    p_af controls class balancing.
    mode: 'fixed' | 'beat' | 'beat_hr'
    """
    def __init__(self, patient_dirs, mode="fixed", p_af=0.5,
                 max_tries=80, seed=0, cache_size=64):
        self.patient_dirs    = list(patient_dirs)
        self.patient_records = [list_record_bases(pd) for pd in self.patient_dirs]
        self.mode            = mode
        self.p_af            = p_af
        self.max_tries       = max_tries
        self.seed            = seed
        self.epoch           = 0
        self._init_cache(cache_size)

    def __len__(self):
        return len(self.patient_dirs)

    def _init_cache(self, cache_size):
        @lru_cache(maxsize=cache_size)
        def _load(rec_str):
            sig = load_signal(rec_str)
            ann = wfdb.rdann(rec_str, extension="atr")
            ivs = build_rhythm_intervals_from_ann(ann, len(sig))
            return sig, ann, ivs
        self._load = _load

    def _sample_window(self, sig, ann, ivs, rng, target_af):
        sig_len = len(sig)
        if sig_len <= WINDOW_SAMPLES:
            return None

        for _ in range(self.max_tries):
            w_start = int(rng.integers(0, sig_len - WINDOW_SAMPLES))
            w_end   = w_start + WINDOW_SAMPLES
            chunk   = sig[w_start:w_end]

            if np.isnan(chunk).any() or np.std(chunk) < 1e-4:
                continue

            lab = window_purity_label(ivs, w_start, w_end, PURITY_THRESH)
            if lab is None:
                continue

            is_af = lab in ("AFIB", "AFL")
            if target_af is not None and is_af != target_af:
                continue

            x = (chunk - chunk.mean()) / (chunk.std() + 1e-6)

            beats, rr = extract_beats_and_rr_from_ann(
                x, ann, w_start, w_end, BEAT_LENGTH
            )
            if beats is None:
                continue

            return x, beats, rr, LABEL2ID[lab]

        return None

    def __getitem__(self, idx):
        rng = np.random.default_rng(
            self.seed + idx * 1000003 + self.epoch * 999983
        )
        target_af = (rng.random() < self.p_af) if self.p_af is not None else None

        recs = self.patient_records[idx]
        if not recs:
            return self.__getitem__((idx + 1) % len(self))

        for _ in range(10):
            rec = recs[int(rng.integers(0, len(recs)))]
            sig, ann, ivs = self._load(str(rec))
            result = self._sample_window(sig, ann, ivs, rng, target_af)
            if result is not None:
                x, beats, rr, y = result
                y = torch.tensor(y, dtype=torch.long)
                if self.mode == "fixed":
                    return torch.from_numpy(x.astype(np.float32)), y
                if self.mode == "beat":
                    return torch.from_numpy(beats), y
                return torch.from_numpy(beats), torch.from_numpy(rr), y

        return self.__getitem__((idx + 1) % len(self))

Building val manifest...
  100/550 patients | 196 windows so far
  200/550 patients | 392 windows so far
  300/550 patients | 590 windows so far
  400/550 patients | 788 windows so far
  500/550 patients | 988 windows so far
Val manifest:  1088 windows

Building test manifest...
  100/550 patients | 192 windows so far
  200/550 patients | 390 windows so far
  300/550 patients | 588 windows so far
  400/550 patients | 784 windows so far
  500/550 patients | 982 windows so far
Test manifest: 1082 windows
Val: N=1088 | AF=83 (7.6%) | NSR=1005 (92.4%)
Test: N=1082 | AF=80 (7.4%) | NSR=1002 (92.6%)


In [32]:
def fixed_collate(batch):
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.stack(ys)


def beat_collate(batch):
    beats_list, ys = zip(*batch)
    B     = len(beats_list)
    max_n = min(max(b.shape[0] for b in beats_list), MAX_BEATS)

    padded = torch.zeros(B, max_n, 1, BEAT_LENGTH)
    mask   = torch.ones(B, max_n, dtype=torch.bool)

    for i, beats in enumerate(beats_list):
        n = min(beats.shape[0], MAX_BEATS)
        padded[i, :n] = beats[:n]
        mask[i, :n]   = False

    return padded, mask, torch.stack(ys)

def beat_hr_collate(batch):
    beats_list, rr_list, ys = zip(*batch)
    B     = len(beats_list)
    max_n = min(max(b.shape[0] for b in beats_list), MAX_BEATS)

    padded = torch.zeros(B, max_n, 1, BEAT_LENGTH)
    padded_rr = torch.zeros(B, max_n)
    mask   = torch.ones(B, max_n, dtype=torch.bool)

    for i, (beats, rr) in enumerate(zip(beats_list, rr_list)):
        n = min(beats.shape[0], MAX_BEATS)
        padded[i, :n]    = beats[:n]
        padded_rr[i, :n] = rr[:n]
        mask[i, :n]      = False

    return padded, padded_rr, mask, torch.stack(ys)

def make_loaders(mode, batch_size=64):
    train_ds = IcentiaTrainDataset(train_patients, mode=mode,
                                   p_af=P_AF_TRAIN, seed=0)

    val_ds   = IcentiaManifestDataset(val_manifest,  mode=mode)
    test_ds  = IcentiaManifestDataset(test_manifest, mode=mode)

    collate_fn = {
        "fixed":   fixed_collate,
        "beat":    beat_collate,
        "beat_hr": beat_hr_collate,
    }[mode]

    kw = dict(batch_size=batch_size, num_workers=8,
              persistent_workers=False, prefetch_factor=2, pin_memory=True)

    train_loader = DataLoader(train_ds, shuffle=True,  collate_fn=collate_fn, **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, collate_fn=collate_fn, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, collate_fn=collate_fn, **kw)

    return train_loader, val_loader, test_loader


fixed_train,   fixed_val,   fixed_test   = make_loaders("fixed")
beat_train,    beat_val,    beat_test    = make_loaders("beat")
beat_hr_train, beat_hr_val, beat_hr_test = make_loaders("beat_hr")

print("Dataloaders ready")

Dataloaders ready


In [33]:
def train_one_epoch_fixed(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def train_one_epoch_beat(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for beats, mask, y in loader:
        beats, mask, y = beats.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(beats, mask), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def train_one_epoch_beat_hr(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for beats, rr, mask, y in loader:
        beats, rr, mask, y = beats.to(device), rr.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(beats, rr, mask), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_beat_hr(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for beats, rr, mask, y in loader:
        beats, rr, mask, y = beats.to(device), rr.to(device), mask.to(device), y.to(device)
        probs = torch.softmax(model(beats, rr, mask), dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    return {
        "AUROC": roc_auc_score(all_labels, all_probs),
        "AUPRC": average_precision_score(all_labels, all_probs),
    }

@torch.no_grad()
def evaluate_fixed(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        probs = torch.softmax(model(x), dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    return {
        "AUROC": roc_auc_score(all_labels, all_probs),
        "AUPRC": average_precision_score(all_labels, all_probs),
    }


@torch.no_grad()
def evaluate_beat(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for beats, mask, y in loader:
        beats, mask, y = beats.to(device), mask.to(device), y.to(device)
        probs = torch.softmax(model(beats, mask), dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    return {
        "AUROC": roc_auc_score(all_labels, all_probs),
        "AUPRC": average_precision_score(all_labels, all_probs),
    }


def run_training(model, train_loader, val_loader, test_loader,
                 train_fn, eval_fn, ckpt_dir,
                 num_epochs=10, lr_enc=1e-4, lr_head=1e-3):

    Path(ckpt_dir).mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": lr_enc},
        {"params": model.classifier.parameters(), "lr": lr_head},
    ], weight_decay=1e-4)

    best_auprc = 0.0
    best_state = None

    for epoch in range(num_epochs):
        train_loader.dataset.epoch = epoch

        train_loss  = train_fn(model, train_loader, optimizer, device)
        val_metrics = eval_fn(model, val_loader, device)

        print(f"Epoch {epoch+1:02d} | loss={train_loss:.4f} | "
              f"val AUROC={val_metrics['AUROC']:.4f}  AUPRC={val_metrics['AUPRC']:.4f}")

        if val_metrics["AUPRC"] > best_auprc:
            best_auprc = val_metrics["AUPRC"]
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, f"{ckpt_dir}/best.pt")
            print(f"  → new best (AUPRC={best_auprc:.4f})")

    model.load_state_dict(best_state)
    test_metrics = eval_fn(model, test_loader, device)
    print(f"\n{'='*50}")
    print(f"TEST  AUROC={test_metrics['AUROC']:.4f}  AUPRC={test_metrics['AUPRC']:.4f}")
    print(f"{'='*50}")
    return test_metrics

In [34]:
# print("Computing beat stats on training patients (this may take a few minutes)...")

# beat_counts     = []   # number of beats per window
# beat_lengths_ms = []   # duration of each beat in samples (R-R interval)

# n_windows_used  = 0
# n_windows_skip  = 0

# for i, pd in enumerate(train_patients):
#     for rec in list_record_bases(pd):
#         try:
#             header    = wfdb.rdheader(str(rec))
#             sig_len   = int(header.sig_len)
#             ann       = wfdb.rdann(str(rec), extension="atr")
#             intervals = build_rhythm_intervals_from_ann(ann, sig_len)
#         except Exception:
#             continue

#         # non-overlapping windows for stats
#         for w_start in range(0, sig_len - WINDOW_SAMPLES + 1, WINDOW_SAMPLES):
#             w_end = w_start + WINDOW_SAMPLES

#             lab = window_purity_label(intervals, w_start, w_end, PURITY_THRESH)
#             if lab is None:
#                 n_windows_skip += 1
#                 continue

#             # get R-peaks in this window
#             r_peaks = sorted([
#                 int(s) for s, sym in zip(ann.sample, ann.symbol)
#                 if w_start <= int(s) < w_end and sym in VALID_BEAT_SYMBOLS
#             ])

#             if len(r_peaks) < 2:
#                 n_windows_skip += 1
#                 continue

#             n_beats = len(r_peaks) - 1
#             beat_counts.append(n_beats)
#             n_windows_used += 1

#             # R-R interval in samples for each beat
#             for j in range(len(r_peaks) - 1):
#                 beat_lengths_ms.append(r_peaks[j+1] - r_peaks[j])

#     if (i + 1) % 20 == 0:
#         print(f"  {i+1}/{len(train_patients)} patients | {n_windows_used} valid windows")

# beat_counts     = np.array(beat_counts)
# beat_lengths_ms = np.array(beat_lengths_ms)

# print(f"\n{'='*55}")
# print(f"Stats over {n_windows_used} valid 60s windows (skipped {n_windows_skip})")
# print(f"{'='*55}")

# print(f"\nBeats per window:")
# print(f"  Min    : {beat_counts.min()}")
# print(f"  Max    : {beat_counts.max()}")
# print(f"  Mean   : {beat_counts.mean():.1f}")
# print(f"  Median : {np.median(beat_counts):.1f}")
# for p in [5, 25, 75, 90, 95, 99]:
#     print(f"  {p}th pct: {np.percentile(beat_counts, p):.1f}")

# print(f"\nMAX_BEATS coverage:")
# for cap in [50, 75, 93, 95, 100, 120]:
#     pct = (beat_counts <= cap).mean() * 100
#     print(f"  MAX_BEATS={cap:3d}: {pct:.1f}% of windows fully covered")

# print(f"\nBeat length in samples (R-R interval at 250Hz):")
# print(f"  Min    : {beat_lengths_ms.min()} samples  ({beat_lengths_ms.min()/FS*1000:.0f} ms)")
# print(f"  Max    : {beat_lengths_ms.max()} samples  ({beat_lengths_ms.max()/FS*1000:.0f} ms)")
# print(f"  Mean   : {beat_lengths_ms.mean():.1f} samples  ({beat_lengths_ms.mean()/FS*1000:.0f} ms)")
# print(f"  Median : {np.median(beat_lengths_ms):.1f} samples  ({np.median(beat_lengths_ms)/FS*1000:.0f} ms)")
# for p in [5, 25, 75, 90, 95, 99]:
#     v = np.percentile(beat_lengths_ms, p)
#     print(f"  {p}th pct: {v:.1f} samples  ({v/FS*1000:.0f} ms)")

# print(f"\nBeat length vs BEAT_LENGTH={BEAT_LENGTH} samples:")
# pct_shorter = (beat_lengths_ms < BEAT_LENGTH).mean() * 100
# pct_longer  = (beat_lengths_ms > BEAT_LENGTH).mean() * 100
# print(f"  Beats shorter than {BEAT_LENGTH} (compressed by resample): {pct_shorter:.1f}%")
# print(f"  Beats longer  than {BEAT_LENGTH} (stretched by resample) : {pct_longer:.1f}%")

### Output ###
# =======================================================
# Stats over 2290396 valid 60s windows (skipped 1459202)
# =======================================================

# Beats per window:
#   Min    : 1
#   Max    : 183
#   Mean   : 68.1
#   Median : 67.0
#   5th pct: 49.0
#   25th pct: 59.0
#   75th pct: 76.0
#   90th pct: 86.0
#   95th pct: 93.0
#   99th pct: 107.0

# MAX_BEATS coverage:
#   MAX_BEATS= 50: 7.4% of windows fully covered
#   MAX_BEATS= 75: 73.8% of windows fully covered
#   MAX_BEATS= 93: 95.4% of windows fully covered
#   MAX_BEATS= 95: 96.3% of windows fully covered
#   MAX_BEATS=100: 97.9% of windows fully covered
#   MAX_BEATS=120: 99.7% of windows fully covered

# Beat length in samples (R-R interval at 250Hz):
#   Min    : 47 samples  (188 ms)
#   Max    : 14017 samples  (56068 ms)
#   Mean   : 216.6 samples  (866 ms)
#   Median : 212.0 samples  (848 ms)
#   5th pct: 145.0 samples  (580 ms)
#   25th pct: 183.0 samples  (732 ms)
#   75th pct: 244.0 samples  (976 ms)
#   90th pct: 276.0 samples  (1104 ms)
#   95th pct: 299.0 samples  (1196 ms)
#   99th pct: 352.0 samples  (1408 ms)

# Beat length vs BEAT_LENGTH=160 samples:
#   Beats shorter than 160 (compressed by resample): 10.2%
#   Beats longer  than 160 (stretched by resample) : 89.3%

In [35]:
def count_labels(loader, name):
    all_labels = []
    for batch in loader:
        y = batch[-1]   # label is always the last element regardless of tokenizer
        all_labels.extend(y.numpy())
    all_labels = np.array(all_labels)
    n_pos = (all_labels == 1).sum()
    n_neg = (all_labels == 0).sum()
    n_tot = len(all_labels)
    print(f"{name:<30} N={n_tot:6d} | AF={n_pos:5d} ({100*n_pos/n_tot:.1f}%) | NSR={n_neg:5d} ({100*n_neg/n_tot:.1f}%)")

print("Label distribution across splits:")
print("-" * 70)
count_labels(fixed_train, "Fixed train")
count_labels(fixed_val,   "Fixed val")
count_labels(fixed_test,  "Fixed test")
print()
count_labels(beat_train,  "Beat-sync train")
count_labels(beat_val,    "Beat-sync val")
count_labels(beat_test,   "Beat-sync test")
print()
count_labels(beat_hr_train, "Beat-sync+HR train")
count_labels(beat_hr_val,   "Beat-sync+HR val")
count_labels(beat_hr_test,  "Beat-sync+HR test")

Label distribution across splits:
----------------------------------------------------------------------
Fixed train                    N=  1100 | AF=  103 (9.4%) | NSR=  997 (90.6%)
Fixed val                      N=  1088 | AF=   83 (7.6%) | NSR= 1005 (92.4%)
Fixed test                     N=  1082 | AF=   80 (7.4%) | NSR= 1002 (92.6%)

Beat-sync train                N=  1100 | AF=  103 (9.4%) | NSR=  997 (90.6%)
Beat-sync val                  N=  1088 | AF=   83 (7.6%) | NSR= 1005 (92.4%)
Beat-sync test                 N=  1082 | AF=   80 (7.4%) | NSR= 1002 (92.6%)

Beat-sync+HR train             N=  1100 | AF=  103 (9.4%) | NSR=  997 (90.6%)
Beat-sync+HR val               N=  1088 | AF=   83 (7.6%) | NSR= 1005 (92.4%)
Beat-sync+HR test              N=  1082 | AF=   80 (7.4%) | NSR= 1002 (92.6%)


In [36]:
print("="*50)
print("FIXED PATCH TOKENIZER (pretrained)")
print("="*50)

fixed_results = run_training(
    model        = fixed_model,
    train_loader = fixed_train,
    val_loader   = fixed_val,
    test_loader  = fixed_test,
    train_fn     = train_one_epoch_fixed,
    eval_fn      = evaluate_fixed,
    ckpt_dir     = "checkpoints_fixed_icentia",
)

FIXED PATCH TOKENIZER (pretrained)
Epoch 01 | loss=0.3113 | val AUROC=0.9160  AUPRC=0.4223
  → new best (AUPRC=0.4223)
Epoch 02 | loss=0.1308 | val AUROC=0.9809  AUPRC=0.7675
  → new best (AUPRC=0.7675)
Epoch 03 | loss=0.0806 | val AUROC=0.9854  AUPRC=0.8640
  → new best (AUPRC=0.8640)
Epoch 04 | loss=0.0392 | val AUROC=0.9859  AUPRC=0.8909
  → new best (AUPRC=0.8909)
Epoch 05 | loss=0.0594 | val AUROC=0.9827  AUPRC=0.8628
Epoch 06 | loss=0.0446 | val AUROC=0.9832  AUPRC=0.8870
Epoch 07 | loss=0.0411 | val AUROC=0.9795  AUPRC=0.8779
Epoch 08 | loss=0.0245 | val AUROC=0.9777  AUPRC=0.8408
Epoch 09 | loss=0.0357 | val AUROC=0.9805  AUPRC=0.8589
Epoch 10 | loss=0.0339 | val AUROC=0.9828  AUPRC=0.8866

TEST  AUROC=0.9879  AUPRC=0.8668


In [37]:
print("="*50)
print("BEAT-SYNC TOKENIZER (pretrained conv weights)")
print("="*50)

beat_results = run_training(
    model        = beat_model,
    train_loader = beat_train,
    val_loader   = beat_val,
    test_loader  = beat_test,
    train_fn     = train_one_epoch_beat,
    eval_fn      = evaluate_beat,
    ckpt_dir     = "checkpoints_beatsync_icentia",
)

BEAT-SYNC TOKENIZER (pretrained conv weights)
Epoch 01 | loss=0.2895 | val AUROC=0.8837  AUPRC=0.3031
  → new best (AUPRC=0.3031)
Epoch 02 | loss=0.1454 | val AUROC=0.9499  AUPRC=0.5919
  → new best (AUPRC=0.5919)
Epoch 03 | loss=0.1228 | val AUROC=0.9650  AUPRC=0.7416
  → new best (AUPRC=0.7416)
Epoch 04 | loss=0.0908 | val AUROC=0.9618  AUPRC=0.7470
  → new best (AUPRC=0.7470)
Epoch 05 | loss=0.1092 | val AUROC=0.9661  AUPRC=0.7561
  → new best (AUPRC=0.7561)
Epoch 06 | loss=0.0921 | val AUROC=0.9768  AUPRC=0.8100
  → new best (AUPRC=0.8100)
Epoch 07 | loss=0.0876 | val AUROC=0.9780  AUPRC=0.8333
  → new best (AUPRC=0.8333)
Epoch 08 | loss=0.0628 | val AUROC=0.9809  AUPRC=0.8412
  → new best (AUPRC=0.8412)
Epoch 09 | loss=0.0582 | val AUROC=0.9752  AUPRC=0.8320
Epoch 10 | loss=0.0488 | val AUROC=0.9833  AUPRC=0.8830
  → new best (AUPRC=0.8830)

TEST  AUROC=0.9908  AUPRC=0.9141


In [38]:
print("="*50)
print("BEAT-SYNC + HR TOKENIZER (Tok3)")
print("="*50)

beat_hr_results = run_training(
    model        = beat_hr_model,
    train_loader = beat_hr_train,
    val_loader   = beat_hr_val,
    test_loader  = beat_hr_test,
    train_fn     = train_one_epoch_beat_hr,
    eval_fn      = evaluate_beat_hr,
    ckpt_dir     = "checkpoints_beatsynchr_icentia",
)

BEAT-SYNC + HR TOKENIZER (Tok3)
Epoch 01 | loss=0.2528 | val AUROC=0.8883  AUPRC=0.3254
  → new best (AUPRC=0.3254)
Epoch 02 | loss=0.1357 | val AUROC=0.9620  AUPRC=0.7034
  → new best (AUPRC=0.7034)
Epoch 03 | loss=0.1157 | val AUROC=0.9746  AUPRC=0.7951
  → new best (AUPRC=0.7951)
Epoch 04 | loss=0.0866 | val AUROC=0.9636  AUPRC=0.7851
Epoch 05 | loss=0.0860 | val AUROC=0.9712  AUPRC=0.7814
Epoch 06 | loss=0.0695 | val AUROC=0.9788  AUPRC=0.8352
  → new best (AUPRC=0.8352)
Epoch 07 | loss=0.0833 | val AUROC=0.9761  AUPRC=0.8289
Epoch 08 | loss=0.0555 | val AUROC=0.9786  AUPRC=0.8514
  → new best (AUPRC=0.8514)
Epoch 09 | loss=0.0528 | val AUROC=0.9843  AUPRC=0.8851
  → new best (AUPRC=0.8851)
Epoch 10 | loss=0.0445 | val AUROC=0.9821  AUPRC=0.8564

TEST  AUROC=0.9811  AUPRC=0.8753


In [39]:
print("\n========== FINAL COMPARISON ==========")
print(f"{'Tokenizer':<25} {'AUROC':>8} {'AUPRC':>8}")
print(f"{'Fixed (p=160)':<25} {fixed_results['AUROC']:>8.4f} {fixed_results['AUPRC']:>8.4f}")
print(f"{'Beat-sync (Tok1)':<25} {beat_results['AUROC']:>8.4f} {beat_results['AUPRC']:>8.4f}")
print(f"{'Beat-sync+HR (Tok3)':<25} {beat_hr_results['AUROC']:>8.4f} {beat_hr_results['AUPRC']:>8.4f}")


========== FINAL COMPARISON ==========
Tokenizer                    AUROC    AUPRC
Fixed (p=160)               0.9879   0.8668
Beat-sync (Tok1)            0.9908   0.9141
Beat-sync+HR (Tok3)         0.9811   0.8753
